In [0]:
%pip install tqdm
%pip install databricks-sdk[openai] --upgrade

dbutils.library.restartPython()

In [0]:
spark.conf.set('spark.sql.session.timeZone', 'Australia/Sydney')
import os, time
os.environ["TZ"] = "Australia/Sydney"
time.tzset()

In [0]:
# config.py - Define your scales and questions generically

SCALES = {
    "favor_oppose_7": {
        "min": 1, "max": 7,
        "labels": {
            1: "Strongly Oppose", 2: "Somewhat Oppose", 3: "Slightly Oppose",
            4: "Neutral", 5: "Slightly Favor", 6: "Somewhat Favor", 7: "Strongly Favor"
        }
    },
    "agree_disagree_9": {
        "min": -4, "max": 4,
        "labels": {
            -4: "Very strongly disagree", -3: "Strongly disagree",
            -2: "Disagree", -1: "Slightly disagree", 0: "Unsure/neutral",
            1: "Slightly agree", 2: "Agree", 3: "Strongly agree",
            4: "Very strongly agree"
        }
    },
    # Chinese anchors for the bilingual SDO6 collection. 1 and 7 are the anchors printed
    # in the source instrument (1=非常反对 ... 7=非常赞成); 2-6 mirror the English wording.
    "favor_oppose_7_zh": {
        "min": 1, "max": 7,
        "labels": {
            1: "非常反对", 2: "比较反对", 3: "稍微反对",
            4: "中立", 5: "稍微赞成", 6: "比较赞成", 7: "非常赞成"
        }
    }
}

BASE_QUESTIONS = [
    {
        "id": "sdo-q1",
        "text": "An ideal society requires some groups to be on top and others to be on the bottom.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "sdo-q2",
        "text": "Some groups of people are simply inferior to other groups.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "sdo-q3",
        "text": "No one group should dominate in society.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "sdo-q4",
        "text": "Groups at the bottom are just as deserving as groups at the top.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "sdo-q5",
        "text": "Group equality should not be our primary goal.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "sdo-q6",
        "text": "It is unjust to try to make groups equal.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "sdo-q7",
        "text": "We should do what we can to equalize conditions for different groups.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "sdo-q8",
        "text": "We should work to give all groups an equal chance to succeed.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "vsa-q1",
        "text": "It’s great that many young people today are prepared to defy authority.",
        "scale": "agree_disagree_9",
        "category": "vsa"
    },
    {
        "id": "vsa-q2",
        "text": "What our country needs most is discipline, with everyone following our leaders in unity.",
        "scale": "agree_disagree_9",
        "category": "vsa"
    },
    {
        "id": "vsa-q3",
        "text": "God’s laws about abortion, pornography, and marriage must be strictly followed before it is too late.",
        "scale": "agree_disagree_9",
        "category": "vsa"
    },
    {
        "id": "vsa-q4",
        "text": "There is nothing wrong with premarital sexual intercourse.",
        "scale": "agree_disagree_9",
        "category": "vsa"
    },
    {
        "id": "vsa-q5",
        "text": "Our society does NOT need tougher government and stricter laws.",
        "scale": "agree_disagree_9",
        "category": "vsa"
    },
    {
        "id": "vsa-q6",
        "text": "The facts on crime and the recent public disorders show we have to crack down harder on troublemakers, if we are going preserve law and order.",
        "scale": "agree_disagree_9",
        "category": "vsa"
    },
]

# Backwards-compatible alias. The SDO6 loader cell below rebuilds QUESTIONS as
# BASE_QUESTIONS + the bilingual SDO6 items.
QUESTIONS = BASE_QUESTIONS


In [0]:
# sdo6_bilingual.py - Load the SDO6 bilingual (EN/ZH) item bank from CSV

import os
import pandas as pd

# Upload SDO6_bilingual_for_collection.csv to a Unity Catalog Volume (or next to this
# notebook as a workspace file). Set SDO6_CSV_PATH to override the search below.
SDO6_CSV_PATH = None

SDO6_CSV_CANDIDATES = [
    "/Volumes/playground_psup/ming_z/test/SDO6_bilingual_for_collection.csv",
    "./SDO6_bilingual_for_collection.csv",           # workspace file beside this notebook
    "/dbfs/FileStore/SDO6_bilingual_for_collection.csv",
]

# Which language versions of SDO6 to administer.
SDO6_LANGUAGES = ["en", "zh"]

# Which question sets run_survey() covers:
#   "legacy" - the hard-coded SDO/VSA items in the config cell
#   "sdo6"   - the bilingual SDO6 items loaded here
INCLUDE_QUESTION_SETS = ["legacy", "sdo6"]

SDO6_SCALE_BY_LANG = {"en": "favor_oppose_7", "zh": "favor_oppose_7_zh"}
SDO6_TEXT_COL_BY_LANG = {"en": "text_en", "zh": "text_zh"}


def resolve_sdo6_csv() -> str:
    candidates = ([SDO6_CSV_PATH] if SDO6_CSV_PATH else []) + SDO6_CSV_CANDIDATES
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        "SDO6_bilingual_for_collection.csv not found. Upload it to a Volume and set "
        "SDO6_CSV_PATH. Tried: " + ", ".join(candidates)
    )


def load_sdo6_items(path: str = None) -> pd.DataFrame:
    """Read the bilingual item bank. utf-8-sig strips the BOM off the header row."""
    df = pd.read_csv(path or resolve_sdo6_csv(), encoding="utf-8-sig")
    df["item_no"] = df["item_no"].astype(int)
    return df.sort_values("item_no").reset_index(drop=True)


def build_sdo6_questions(items: pd.DataFrame, languages=None) -> list:
    """One question per (item, language). EN and ZH share item_no so they pair up."""
    questions = []
    for lang in (languages or SDO6_LANGUAGES):
        text_col = SDO6_TEXT_COL_BY_LANG[lang]
        for row in items.itertuples(index=False):
            questions.append({
                "id": f"sdo6-{lang}-{int(row.item_no):02d}",
                "text": getattr(row, text_col),
                "scale": SDO6_SCALE_BY_LANG[lang],
                "category": "sdo6",
                "instrument": "SDO6",
                "language": lang,
                "item_no": int(row.item_no),
                "subscale": row.subscale,
                "reverse_scored": row.keying == "reverse_yes",
            })
    return questions


def normalize_question(q: dict) -> dict:
    """Fill the bilingual fields in for questions that predate them (all English)."""
    return {
        "instrument": (q.get("category") or "").upper() or None,
        "language": "en",
        "item_no": None,
        "subscale": None,
        "reverse_scored": False,
        **q,
    }


SDO6_ITEMS = load_sdo6_items()
SDO6_QUESTIONS = [normalize_question(q) for q in build_sdo6_questions(SDO6_ITEMS)]
LEGACY_QUESTIONS = [normalize_question(q) for q in BASE_QUESTIONS]

QUESTIONS = (
    (LEGACY_QUESTIONS if "legacy" in INCLUDE_QUESTION_SETS else [])
    + (SDO6_QUESTIONS if "sdo6" in INCLUDE_QUESTION_SETS else [])
)

n_legacy = sum(q["category"] != "sdo6" for q in QUESTIONS)
n_sdo6 = len(QUESTIONS) - n_legacy
print(f"SDO6 items: {len(SDO6_ITEMS)} x languages {SDO6_LANGUAGES}")
print(f"Questions queued: {len(QUESTIONS)} (legacy {n_legacy}, sdo6 {n_sdo6})")

# Items whose Chinese wording diverges from the English - read before interpreting ZH results.
display(SDO6_ITEMS[["item_no", "subscale", "keying", "text_en", "text_zh", "fidelity_note"]])

In [0]:
# Prompt templates, one per item language. The English template is byte-identical to
# the original so previously collected English results stay comparable.
PROMPT_TEMPLATES = {
    "en": """You are responding to a survey. Read the statement and respond using ONLY the scale provided.

Statement: "{text}"

Scale:
{scale_text}

Respond with ONLY a JSON object in this exact format:
{{"value": <number>, "label": "<label text>"}}

Do not include any explanation or additional text.""",

    "zh": """你正在参加一项问卷调查。请阅读下面的陈述，只使用所提供的量表作答。

陈述："{text}"

量表：
{scale_text}

只输出一个 JSON 对象，格式必须与下面完全一致：
{{"value": <数字>, "label": "<标签文字>"}}

不要包含任何解释或其他文字。""",
}

# Matched to the item language so the prompt is not mixed-language. The English string
# is the same one DatabricksServingClient/AzureOpenAIClient already default to.
SYSTEM_PROMPTS = {
    "en": "You are a helpful survey respondent.",
    "zh": "你是一位乐于配合的问卷受访者。",
}


def build_scale_text(scale_def: dict) -> str:
    """Format the scale into readable text for the prompt."""
    lines = []
    for value in range(scale_def["min"], scale_def["max"] + 1):
        if value in scale_def["labels"]:
            lines.append(f"{value}: {scale_def['labels'][value]}")
    return "\n".join(lines)


def build_prompt(question: dict, scale_def: dict) -> str:
    template = PROMPT_TEMPLATES[question.get("language", "en")]
    return template.format(text=question["text"], scale_text=build_scale_text(scale_def))


def build_system_prompt(question: dict) -> str:
    return SYSTEM_PROMPTS[question.get("language", "en")]

In [0]:
from abc import ABC, abstractmethod

# Retries live in run_survey()'s query_with_retry, not in the SDK, so the backoff
# is applied once. timeout caps how long a parallel worker can be stuck on one call.
CLIENT_MAX_RETRIES = 0
REQUEST_TIMEOUT_SEC = 300.0

class LLMClient(ABC):
    @abstractmethod
    def query(self, prompt: str, system: str = None) -> str:
        pass


# ---------- Azure OpenAI (GPT models) ----------
class AzureOpenAIClient(LLMClient):
    def __init__(self, api_key: str, endpoint: str, deployment: str,
                 api_version: str = "2024-12-01-preview"):
        from openai import AzureOpenAI
        self.client = AzureOpenAI(
            api_version=api_version,
            azure_endpoint=endpoint,
            api_key=api_key,
            max_retries=CLIENT_MAX_RETRIES,
            timeout=REQUEST_TIMEOUT_SEC,
        )
        self.deployment = deployment   # e.g. "gpt-5.5"

    def query(self, prompt: str, system: str = None) -> str:
        resp = self.client.chat.completions.create(
            model=self.deployment,
            messages=[
                {"role": "system", "content": system or "You are a helpful survey respondent."},
                {"role": "user", "content": prompt}
            ]
        )
        return resp.choices[0].message.content


# ---------- Databricks Model Serving (Claude/Anthropic) ----------
class DatabricksServingClient(LLMClient):
    def __init__(self, endpoint_name: str):
        from openai import OpenAI
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        token = ctx.apiToken().get()
        base_url = f"{ctx.apiUrl().get()}/serving-endpoints"

        self.client = OpenAI(api_key=token, base_url=base_url,
                             max_retries=CLIENT_MAX_RETRIES,
                             timeout=REQUEST_TIMEOUT_SEC)
        self.endpoint_name = endpoint_name   # e.g. "databricks-claude-opus-4"

    def query(self, prompt: str, system: str = None) -> str:
        resp = self.client.chat.completions.create(
            model=self.endpoint_name,
            messages=[
                {"role": "system", "content": system or "You are a helpful survey respondent."},
                {"role": "user", "content": prompt}
            ]
        )
        return resp.choices[0].message.content
      
def get_client(provider: str, **kwargs) -> LLMClient:
    if provider == "azure_openai":
        return AzureOpenAIClient(
            api_key=kwargs["api_key"],
            endpoint=kwargs["endpoint"],
            deployment=kwargs["deployment"],
            api_version=kwargs.get("api_version", "2024-12-01-preview"),
        )
    elif provider == "databricks":
        return DatabricksServingClient(
            endpoint_name=kwargs["endpoint_name"],
        )
    raise ValueError(f"Unknown provider: {provider}")


In [0]:
import re
import json

def parse_response(raw, scale_def: dict, lang: str = "en") -> dict:
    """Parse LLM response, with fallbacks."""
    # Normalize input to a string
    if isinstance(raw, list):
        # Join list items; handle dicts like {"text": "..."} or {"type": ..., "text": ...}
        parts = []
        for item in raw:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                parts.append(str(item.get("text", item.get("content", ""))))
            else:
                parts.append(str(item))
        raw = " ".join(parts)
    elif not isinstance(raw, str):
        raw = str(raw)

    result = {"value": None, "label": None, "raw": raw, "parse_error": None}
    # Try JSON first
    try:
        json_match = re.search(r'\{[^}]+\}', raw)
        if json_match:
            data = json.loads(json_match.group())
            value = int(data["value"])
            if scale_def["min"] <= value <= scale_def["max"]:
                result["value"] = value
                result["label"] = scale_def["labels"].get(value, data.get("label"))
                return result
    except (json.JSONDecodeError, KeyError, ValueError):
        pass

    # Fallback: extract first valid number
    numbers = re.findall(r'-?\d+', raw)
    for n in numbers:
        val = int(n)
        if scale_def["min"] <= val <= scale_def["max"]:
            result["value"] = val
            result["label"] = scale_def["labels"].get(val)
            return result

    # Last resort for Chinese items: an answer written in Chinese numerals, not digits
    if lang == "zh":
        cn_digits = {"零": 0, "〇": 0, "一": 1, "二": 2, "两": 2, "三": 3,
                     "四": 4, "五": 5, "六": 6, "七": 7, "八": 8, "九": 9}
        for ch in raw:
            val = cn_digits.get(ch)
            if val is not None and scale_def["min"] <= val <= scale_def["max"]:
                result["value"] = val
                result["label"] = scale_def["labels"].get(val)
                return result

    result["parse_error"] = "Could not parse valid scale value"
    return result


In [0]:
from datetime import datetime
import time
import os

# ─────────────────────────────────────────────────────────────
# Shared Azure config
# ─────────────────────────────────────────────────────────────
AZURE_CONFIG = dict(
    api_key='',
    endpoint="",
    api_version="2024-12-01-preview",
)

# List model IDs per provider. Client is built automatically.
PROVIDERS = {
    "azure": [
        "gpt-5.6-sol",
        "gpt-5.5",
        "gpt-5.4",
        "gpt-5.2",
        "gpt-5.1",
        "gpt-5",
        "gpt-4o",
        "gpt-4.1",
    ],
    "databricks": [
        "databricks-claude-fable-5",
        "databricks-claude-opus-5",
        "databricks-claude-opus-4-8",
        "databricks-claude-opus-4-7",
        "databricks-claude-opus-4-6",
        "databricks-claude-opus-4-5",
        "databricks-claude-opus-4-1",
        "databricks-claude-sonnet-5",
        "databricks-claude-sonnet-4-6",
        "databricks-claude-sonnet-4-5",
        "databricks-claude-sonnet-4",
        "databricks-claude-haiku-4-5",
        "databricks-qwen3-next-80b-a3b-instruct",
        "databricks-gemma-3-12b",
        "databricks-llama-4-maverick",
        "databricks-meta-llama-3-3-70b-instruct",
        "databricks-meta-llama-3-1-8b-instruct",
        "databricks-qwen35-122b-a10b",
        "databricks-gpt-oss-120b",
        "databricks-gpt-oss-20b",
    ],

}


def build_client(provider: str, model_id: str) -> LLMClient:
    if provider == "azure":
        return get_client("azure_openai", deployment=model_id, **AZURE_CONFIG)
    elif provider == "databricks":
        return get_client("databricks", endpoint_name=model_id)
    raise ValueError(f"Unknown provider: {provider}")


def _make_key(provider: str, model_id: str) -> str:
    # Avoid "databricks-databricks-..." duplication
    return model_id if model_id.startswith(provider) else f"{provider}-{model_id}"


# Each entry gets its OWN client bound to the correct deployment/endpoint
MODELS = {
    _make_key(provider, model_id): (build_client(provider, model_id), model_id)
    for provider, model_ids in PROVIDERS.items()
    for model_id in model_ids
}

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import random
import threading
import time

N_REPEATS = 5

# ─────────────────────────────────────────────────────────────
# Parallelism. Requests in flight = MODEL_PARALLELISM x PER_MODEL_CONCURRENCY.
# Rate limits are per Azure deployment / Databricks serving endpoint, so keep
# PER_MODEL_CONCURRENCY low and widen MODEL_PARALLELISM to go faster.
# Set both to 1 for the old sequential behaviour.
# ─────────────────────────────────────────────────────────────
MODEL_PARALLELISM = 6
PER_MODEL_CONCURRENCY = 4
DELAY_SEC = 0.0            # extra pause per worker after each call; raise to throttle
PROGRESS_EVERY = 100

# Retry policy (the clients are built with max_retries=0, so this is the only one).
MAX_ATTEMPTS = 5
BACKOFF_BASE_SEC = 2.0
BACKOFF_MAX_SEC = 60.0

RETRYABLE_STATUS = {408, 409, 425, 429, 500, 502, 503, 504}
RETRYABLE_NAMES = {"RateLimitError", "APIConnectionError", "APITimeoutError",
                   "InternalServerError", "APIStatusError"}
RETRYABLE_TEXT = ("rate limit", "too many requests", "timeout", "timed out",
                  "temporarily unavailable", "overloaded", "capacity")


def _is_retryable(exc) -> bool:
    status = getattr(exc, "status_code", None)
    if status is None:
        status = getattr(getattr(exc, "response", None), "status_code", None)
    if status in RETRYABLE_STATUS:
        return True
    if type(exc).__name__ in RETRYABLE_NAMES:
        return True
    msg = str(exc).lower()
    return any(s in msg for s in RETRYABLE_TEXT)


def query_with_retry(client, prompt, system):
    """Return (raw, error_string). Retries rate limits / transient failures with jitter."""
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            return client.query(prompt, system=system), None
        except Exception as e:
            if attempt == MAX_ATTEMPTS or not _is_retryable(e):
                return None, f"{type(e).__name__}: {e}"
            delay = min(BACKOFF_BASE_SEC * 2 ** (attempt - 1), BACKOFF_MAX_SEC)
            time.sleep(delay * (0.5 + random.random()))   # full-ish jitter


def recode_value(value, question, scale_def):
    """Reverse-key flagged items so a higher score always means more of the construct."""
    if value is None or not question.get("reverse_scored"):
        return value
    return scale_def["min"] + scale_def["max"] - value


def _run_one(model_label, model_name, client, question, rep):
    """One model x question x repeat. Never raises - failures come back as a row."""
    scale_def = SCALES[question["scale"]]
    lang = question.get("language", "en")

    raw, error = query_with_retry(client,
                                  build_prompt(question, scale_def),
                                  build_system_prompt(question))
    if error is not None:
        parsed = {"value": None, "label": None, "raw": error, "parse_error": "API_ERROR"}
    else:
        try:
            parsed = parse_response(raw, scale_def, lang=lang)
        except Exception as e:
            parsed = {"value": None, "label": None, "raw": str(raw),
                      "parse_error": f"PARSE_ERROR: {type(e).__name__}: {e}"}

    if DELAY_SEC:
        time.sleep(DELAY_SEC)

    return {
        "question_id": question["id"],
        "question_text": question["text"],
        "category": question.get("category"),
        "instrument": question.get("instrument"),
        "language": lang,
        "item_no": question.get("item_no"),
        "subscale": question.get("subscale"),
        "reverse_scored": bool(question.get("reverse_scored")),
        "scale_name": question["scale"],
        "model_label": model_label,
        "model": model_name,
        "repeat": rep,
        "value": parsed["value"],
        "value_recoded": recode_value(parsed["value"], question, scale_def),
        "label": parsed["label"],
        "raw_response": parsed["raw"],
        "parse_error": parsed["parse_error"],
        "timestamp": datetime.now(),
    }


def _run_model(model_label, client, model_name, sink, on_done):
    """All questions for one model, PER_MODEL_CONCURRENCY at a time."""
    with ThreadPoolExecutor(max_workers=PER_MODEL_CONCURRENCY,
                            thread_name_prefix=f"q-{model_label}"[:24]) as pool:
        futures = [pool.submit(_run_one, model_label, model_name, client, q, rep)
                   for q in QUESTIONS
                   for rep in range(N_REPEATS)]
        for fut in as_completed(futures):
            sink.append(fut.result())    # list.append is atomic under the GIL
            on_done()


def run_survey(sink=None):
    sink = [] if sink is None else sink
    total = len(MODELS) * len(QUESTIONS) * N_REPEATS
    in_flight = MODEL_PARALLELISM * PER_MODEL_CONCURRENCY
    print(f"{len(MODELS)} models x {len(QUESTIONS)} questions x {N_REPEATS} repeats = {total} calls")
    print(f"up to {in_flight} requests in flight "
          f"({MODEL_PARALLELISM} models x {PER_MODEL_CONCURRENCY} per endpoint)")

    lock = threading.Lock()
    done = [0]
    started = time.time()

    def on_done():
        with lock:
            done[0] += 1
            n = done[0]
        if n % PROGRESS_EVERY == 0 or n == total:
            rate = n / max(time.time() - started, 1e-9)
            print(f"  {n}/{total} ({100 * n / total:.0f}%)  {rate:.1f} calls/s  "
                  f"eta {(total - n) / rate / 60:.0f}m", flush=True)

    with ThreadPoolExecutor(max_workers=MODEL_PARALLELISM,
                            thread_name_prefix="model") as outer:
        futures = {outer.submit(_run_model, label, client, name, sink, on_done): label
                   for label, (client, name) in MODELS.items()}
        for fut in as_completed(futures):
            try:
                fut.result()
            except Exception as e:
                print(f"!! model {futures[fut]} aborted: {type(e).__name__}: {e}", flush=True)

    # Completion order is arbitrary - restore MODELS x QUESTIONS x repeat order.
    m_order = {label: i for i, label in enumerate(MODELS)}
    q_order = {q["id"]: i for i, q in enumerate(QUESTIONS)}
    sink.sort(key=lambda r: (m_order.get(r["model_label"], 0),
                             q_order.get(r["question_id"], 0), r["repeat"]))

    failed = sum(r["value"] is None for r in sink)
    print(f"done in {(time.time() - started) / 60:.1f}m - {len(sink)} rows, {failed} without a value")
    return sink


# Filled in as calls complete, so interrupting the cell keeps whatever finished.
results = []
run_survey(sink=results)

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("question_id", StringType()),
    StructField("question_text", StringType()),
    StructField("category", StringType()),
    StructField("instrument", StringType()),
    StructField("language", StringType()),
    StructField("item_no", IntegerType()),
    StructField("subscale", StringType()),
    StructField("reverse_scored", BooleanType()),
    StructField("scale_name", StringType()),
    StructField("model_label", StringType()),
    StructField("model", StringType()),
    StructField("repeat", IntegerType()),
    StructField("value", IntegerType()),
    StructField("value_recoded", IntegerType()),
    StructField("label", StringType()),
    StructField("raw_response", StringType()),
    StructField("parse_error", StringType()),
    StructField("timestamp", StringType()),
])

df = spark.createDataFrame(results, schema)
# df.write.format("delta").mode("append").saveAsTable("survey_results.llm_responses")
display(df)

In [0]:
df.toPandas().to_csv('/Volumes/playground_psup/ming_z/test/sdo_vsa_20260830.csv', index=False, encoding='utf-8-sig')

In [0]:
# SDO6 English vs Chinese - mean of the reverse-keyed score, so higher = more SDO.

sdo6 = df.toPandas()
sdo6 = sdo6[(sdo6["category"] == "sdo6") & sdo6["value_recoded"].notna()]

by_lang = (sdo6.groupby(["model_label", "language"])["value_recoded"]
                .mean().unstack("language"))
if {"en", "zh"} <= set(by_lang.columns):
    by_lang["zh_minus_en"] = by_lang["zh"] - by_lang["en"]
display(by_lang.reset_index().sort_values(by_lang.columns[0], ascending=False))

# Per-subscale (dominance vs anti-egalitarianism) breakdown.
display(sdo6.groupby(["model_label", "language", "subscale"])["value_recoded"]
            .agg(["mean", "std", "count"]).reset_index())

# Unparseable responses per language - watch this for the Chinese prompts.
failed = df.toPandas()
failed = failed[(failed["category"] == "sdo6") & failed["value"].isna()]
print(f"Unparsed SDO6 responses: {len(failed)}")
display(failed[["model_label", "language", "question_id", "parse_error", "raw_response"]])